# Simplified Copy-Trade Backtest

Simulates copying trades of a hand-picked set of wallets against the
processed Polygon trade shards.

## Strategy
- When a watched wallet prints a **BUY** on token `(condition_id, outcome)` at price `p`
  we place a buy order at a **slightly worse** price: `order_price = clip(p + WORSE_PRICE_OFFSET, 0.001, 0.999)`.
- The order is matched against the **first subsequent trade** whose effective price is
  `<= order_price` within `FILL_HORIZON_SECONDS`.
  - *same-token* liquidity: later BUY trades on the same `(condition_id, outcome)`.
  - *opposite-token* liquidity: later SELL trades on the **complementary** outcome
    (binary markets only), with effective price = `1 - sell_price`.
- For a wallet **SELL** the mirror applies: order at `p - WORSE_PRICE_OFFSET`, filled
  by the first later trade with effective price `>= order_price`.
  - same-token: later SELL prints on the same outcome.
  - opposite-token: later BUY prints on the complementary outcome, effective price `1 - buy_price`.

## Sharding
Trades are partitioned by the first hex character of `condition_id` (after `0x`).
All shards are processed in parallel; within each shard the backtest is exact
because a `condition_id` always falls in exactly one shard file.

## Outputs
One event ledger DataFrame with columns:
- `row_type` — `trigger` (watched-wallet trade that generated an order) or `fill` (our fill).
- `order_id` — unique integer identifying the copy order.
- `wallet` — originating wallet (`fill` rows carry the trigger wallet for reference).
- `condition_id`, `outcome`, `side`, `dt`, `tx_hash`, `price`.
- `order_price` — the price our order was placed at (`NaN` for fill rows).
- `fill_source` — `same_token` / `opposite_token` / `NaN` for trigger rows.
- `token_winner` — market resolution flag.
- `fill_pnl_usdc` — PnL of *our* position on fill rows (NaN for trigger rows),
  computed as stake × ( 1/exec_price − 1 ) for winning tokens, −stake for losers.
- `filled_wallet_total_position` — total filled position for this wallet across all conditions/outcomes.
- `filled_token_total_position` — total filled position for this `(condition_id, outcome)` across all wallets.
- `filled_token_by_wallet_position` — filled position for this `(wallet, condition_id, outcome)`.

## Visualisation
Cumulative PnL chart with two series:
- **Wallet cohort** — raw `trade_pnl` of the watched wallets.
- **Copy strategy** — `fill_pnl_usdc` of our simulated fills.


## Configuration

In [158]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

# ── Input wallets ─────────────────────────────────────────────────────────────
# Replace with any wallet addresses you want to copy.
# Example: load from a CSV, a workspace registry, or define inline.
WATCHED_WALLETS: list[str] = None

ONLY_BUY_TRIGGERS = False
MAX_EXPOSURE_PER_WALLET_1h = 200

# ── Test period ───────────────────────────────────────────────────────────────
# None = use entire dataset (start/end are inferred from the data).
# Set to datetime.date objects to restrict the window.
PERIOD_START: datetime.date | None = datetime.date(2026, 4, 16)
# PERIOD_START: datetime.date | None = datetime.date(2026, 1, 1) # None   # e.g. datetime.date(2026, 2, 16)
PERIOD_END:   datetime.date | None = datetime.date(2026, 6, 20)
# PERIOD_END:   datetime.date | None = datetime.date(2026, 2, 16) # None   # e.g. datetime.date(2026, 3, 11)

# ── Copy-trade execution parameters ──────────────────────────────────────────
WORSE_PRICE_OFFSET: float = 0
FILL_HORIZON_SECONDS: float = 300     # max seconds after trigger to search for a fill
STAKE_USDC: float = 10.0               # max USDC per order (order qty = min(trigger_qty, STAKE_USDC / order_price))
FEE_BPS: float = 10.0                   # taker fee in basis points
MAX_POSITION_PER_CONDITION: float | None = 1000  # max qty per (wallet, condition_id) across all outcomes; None = unlimited
MAX_POSITION_PER_WALLET:    float | None = 100  # max qty per wallet across all conditions; None = unlimited

# ── Data ─────────────────────────────────────────────────────────────────────
TRADES_DIR     = Path("../../data/polygon_trades_processed")
RAW_TRADES_DIR = Path("../../data/trades_polygon")

# ── Parallelism ───────────────────────────────────────────────────────────────
MAX_WORKERS: int = 10   # number of threads for parallel shard processing

WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
wallets_path = WORKSPACE_DIR / "selected_wallets_v2.parquet"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [159]:
CONDITION='0xfbeac45ae53828674baf51e12313b7fb651594915cfb082e7cc8694896f6fc21'


# df = pd.read_parquet(TRADES_DIR / (CONDITION[2] + ".parquet"))
df = pd.concat(pd.read_parquet(p) for p in TRADES_DIR.glob("*.parquet"))
df = df[df['is_train'] == False].copy()

In [160]:
print(len(df[df['condition_id'] == CONDITION]))
len(df)

191


5136962

In [161]:
wdf = pd.read_parquet(wallets_path)

In [162]:
wdf.head()

,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,...,sample_mult,downside_tail,predictability_score,predicting_final_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score,wallet_quality,wallet_group
0,0x3e438fdb437d90925b47b6f8ab85405ba8ffa83f,0.988890,56,34,66,5978.575157,1037.159066,644.303983,0.538294,0.787085,...,0.531917,0.269878,-0.582112,-0.582112,0.107751,0.114708,4.314947,1.376712,1.376712,copyable
1,0xb78ddd792ad7d1f603fcd046c662d6d9adbf835d,1.520942,124,67,128,13154.695188,2148.207585,1169.647364,0.433823,0.675410,...,0.635229,0.342110,-1.192221,-1.192221,0.088908,0.091842,2.915341,0.450804,0.450804,copyable
2,0x99984e22205053950eb25453779267bcc1aee858,0.354506,3292,1031,3930,32526.435485,3551.573496,491.516453,0.211501,0.329893,...,1.000000,0.138185,-0.491633,-0.491633,0.015111,0.187128,0.745502,0.003221,0.003221,copyable
3,0x22dbd51e1a4e10fff072fa24300238c64033190f,0.702413,10418,2573,12969,702549.388672,44426.924398,11371.920096,0.189942,0.318609,...,1.000000,0.110562,-0.679026,-0.679026,0.016187,0.231426,0.565202,-0.181335,-0.181335,copyable
4,0xe97b7b1396a99341b32984fa8bbf62f2f68ab2ea,1.362514,221,47,262,50819.144170,6107.897999,1358.211608,0.396656,0.622696,...,0.710794,0.212643,-0.996668,-0.996668,0.026726,0.330738,1.038844,-0.182463,-0.182463,copyable


In [163]:
copyable_set = set(wdf[wdf['wallet_group'] == 'copyable']['wallet'].to_list())
watched_set = set(wdf['wallet'].to_list())

In [164]:
df[df['condition_id'] == '0xf25804c3fe472903e37da74c0e80520c397e6be15a55013d3c7d39117adeb783'].sort_values('dt', ascending=False).head(20)

,wallet,condition_id,dt,side,outcome,position,total_quantity,avg_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count


In [165]:
df['is_copyable_wallet'] = df['wallet'].isin(copyable_set)
print(f"Copyable wallets: {len(copyable_set)}, copyable fills: {df['is_copyable_wallet'].sum()} total pnl: {df[df['is_copyable_wallet']]['trade_pnl'].sum():,.2f}")
df['is_watched_wallet'] = df['wallet'].isin(watched_set)
print(f"Watched wallets: {len(watched_set)}, watched fills: {df['is_watched_wallet'].sum()} total pnl: {df[df['is_watched_wallet']]['trade_pnl'].sum():,.2f}")

Copyable wallets: 58, copyable fills: 29184 total pnl: 241,488.69
Watched wallets: 81, watched fills: 73816 total pnl: 404,516.13


In [166]:
df[df['is_copyable_wallet'] == True]

,wallet,condition_id,dt,side,outcome,position,total_quantity,avg_price,trade_value_usdc,final_value_usdc,...,token_winner,final_price,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count,is_copyable_wallet,is_watched_wallet
48316,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0x85ee39382ea9499a4667681e020afa52d496207a2fbcfed7154792457e57b876,2026-04-17 14:20:50+00:00,BUY,Yes,175.270000,175.27,0.521000,91.31567,175.27,...,True,1.0,0x53e66240932d0c3428be3961fd713cf3c9627b44335af5931011b2c8fa4db8bc,1,False,-3.552714e-14,2.664535e-15,0.0,True,True
48317,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0x8a0da22bb30a5af1376ef39624155bef82e4e2f9a1a3b3ed629a43db15a3a22c,2026-04-17 17:31:34+00:00,BUY,Yes,300.000000,300.00,0.290000,87.00000,300.00,...,True,1.0,0x1dbe0b8ebbf4f95dbcb4768afa13c6b5f3acd7346ee261724a0250f30d83380d,1,False,-1.776357e-15,4.440892e-16,0.0,True,True
48318,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0x87d10b2fae256d634cfd9ceb7ed03cb4244830e191a35299be965e53f903ac16,2026-04-17 17:40:32+00:00,BUY,Yes,1487.291426,58.72,0.180000,10.56960,58.72,...,True,1.0,0x2fd5c6c35cf221c86a37f06225b26bc05b025f957c13b9b352bc95fe40a28133,1,False,0.000000e+00,0.000000e+00,0.0,True,True
48319,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0x87d10b2fae256d634cfd9ceb7ed03cb4244830e191a35299be965e53f903ac16,2026-04-17 22:49:40+00:00,SELL,Yes,1387.291426,234.00,0.217009,50.78000,234.00,...,True,1.0,0xd865b9a471ea8abf5084fffdc26c9c83dfa361f7c0eadf31ac01e2452d2d42f1,6,False,1.364242e-12,7.460699e-14,0.0,True,True
48320,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0x87d10b2fae256d634cfd9ceb7ed03cb4244830e191a35299be965e53f903ac16,2026-04-17 23:48:06+00:00,BUY,Yes,1311.621426,58.33,0.090000,5.24970,58.33,...,True,1.0,0x8b42d4393cd1644a4987ee6aaf24db7529da180450316e726de9b60ca21304ce,1,False,0.000000e+00,0.000000e+00,0.0,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1181848,0xe732156a2d84cdfb4de831d3f11a22899e49898f,0x42973552f8bcfc9269fba8492766f2c20a22243cbba154cf695acaff88ec96c7,2026-06-04 00:10:39+00:00,SELL,Yes,0.005453,177.23,0.560000,99.24880,177.23,...,True,1.0,0xef9de55be2bb4baca769ce3ac0852e7d1e8111a46344f806b46eda3971f25de7,1,False,1.772300e+02,1.125864e+03,36.0,True,True
1181849,0xe732156a2d84cdfb4de831d3f11a22899e49898f,0x455e00ee88dbcab92161e3e6d5e5d8e109914536dd577c317c839aeff389e287,2026-06-13 16:00:51+00:00,BUY,Yes,30.000000,30.00,0.987000,29.61000,30.00,...,True,1.0,0x7264a83582dcac9429a27a637c1d5c105ada190f353f7e40b64a6be7e311bdcd,1,False,-2.842171e-14,-2.664535e-15,0.0,True,True
1181850,0xe732156a2d84cdfb4de831d3f11a22899e49898f,0x4db4d5c746da3cb591a9299082c7d4a70fba42ee069fcc7423b90def92e19647,2026-06-14 05:00:27+00:00,BUY,Yes,20.000000,20.00,0.690000,13.80000,20.00,...,True,1.0,0xac6672690c3d9e12de931ba416040faaf09ac5d9baca4aecbd4ee5f149e483cc,1,False,5.000000e+00,3.400000e+00,1.0,True,True
1181851,0xe732156a2d84cdfb4de831d3f11a22899e49898f,0x4db4d5c746da3cb591a9299082c7d4a70fba42ee069fcc7423b90def92e19647,2026-06-14 05:36:27+00:00,SELL,Yes,0.000000,20.00,0.690000,13.80000,20.00,...,True,1.0,0xb5aa7bbd901db2976726ab1bbf8b6d814fdf36bd88b0f67be8a81a4084a97071,1,False,2.000000e+01,1.200000e+02,4.0,True,True


In [167]:
(
    df.assign(
        watched_wallet_pnl=df["trade_pnl"] * df["is_watched_wallet"],
        copyable_wallet_pnl=df["trade_pnl"] * df["is_copyable_wallet"]
    )
    .groupby("condition_id")
    .agg(
        fills_count=("condition_id", "count"),
        wallet_set_fills_count=("is_watched_wallet", "sum"),
        copyable_wallet_set_fills_count=("is_copyable_wallet", "sum"),
        watched_wallets_pnl=("watched_wallet_pnl", "sum"),
        copyable_wallets_pnl=("copyable_wallet_pnl", "sum"),
    )
    .sort_values("wallet_set_fills_count", ascending=False)
    .iloc[0:]
    .head(20)
).sum()

fills_count                        410597.000000
wallet_set_fills_count               9231.000000
copyable_wallet_set_fills_count      7341.000000
watched_wallets_pnl                203517.899719
copyable_wallets_pnl               135424.778783
dtype: float64

In [168]:
df[df['wallet'].isin(copyable_set)].groupby("condition_id")['condition_id'].count().sort_values(ascending=False).head(20)

condition_id
0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423    1095
0x924a2942747dd75703321a7c8d809c68f6a514c3b0f2a2e64274e02310634669     818
0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cdc09e9d47e7d54356c6cd     617
0xa573400b588079857899fce1e5a68da2d086f63fd35cbdd626df397e19e9995e     514
0xc0be7b1f19f9b658778c2be7e6bc67596a00f347ab64392d0f5d387534c7c3b4     392
0x6a8cfe84d17693425f27831db5949d7511f3393d4624b182ac6956164cd32b10     390
0x2135ffcb43ba9103bb6acf7116d2d5aa98bef6d9eb3dc9c85ea00cb79513f3ec     365
0xa48ee32a0cbe5bc1a48844bd14e1691701d3bf8f45f4dd8cb8d1d304561393b6     357
0xaa145cf5714455f546afe53037b8712df749ba96c04820c2d41f37f877f29697     355
0x16608c4fdd7cb41292f6d42c1c02b43ac22593f1c198a104847a3479848d7abe     337
0x421bc1929df1429cf2cb94f80c1ce6a3ed0d1f0b7a2749b9890075f94eb549e9     328
0xb7dd41c16cd5b59e543ffcdab6a0d876f5ccef5329ea42ef1798d80c9a2b8499     319
0x5e5482b1e2afa7bed0b9a0c3c8ad33a83dd6cc6d649666cdf1977be64c42c82a     291
0xceb6dfaa2c

In [169]:
df.groupby("condition_id")['condition_id'].count().sort_values(ascending=False).head(20)

condition_id
0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423    84280
0x924a2942747dd75703321a7c8d809c68f6a514c3b0f2a2e64274e02310634669    48276
0xceb6dfaa2cf5abc9d47ebc867b984a7715104944249274e8a483a2e17473e5f5    39505
0xbbc6689d0f6d57ea42168836712237c7308b3e0118c8914d31b6126d0f3254c5    36864
0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cdc09e9d47e7d54356c6cd    34170
0xc0be7b1f19f9b658778c2be7e6bc67596a00f347ab64392d0f5d387534c7c3b4    26702
0xb7dd41c16cd5b59e543ffcdab6a0d876f5ccef5329ea42ef1798d80c9a2b8499    25835
0xf9f0965a3bd704ad9c184851242606ad35a4ffe733febf5074d4fbcc59d52eb8    24629
0x20af55ab35186377b81219db6cb8615240cba42cea41731091be9484a5f5b122    24125
0x421bc1929df1429cf2cb94f80c1ce6a3ed0d1f0b7a2749b9890075f94eb549e9    23319
0xa573400b588079857899fce1e5a68da2d086f63fd35cbdd626df397e19e9995e    21760
0x1db02ba50e2312a62b4104de691cc7a76065d8d0da40decf93eb1b914a3217b7    20988
0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9f6dc8f59dea51df6c734    19458

In [170]:
banned_wallets = set()
if(WATCHED_WALLETS is not None):
    WATCHED_WALLETS = [w for w in WATCHED_WALLETS if w not in banned_wallets]

## Optionally load wallets from stage-1 workspace

If `WATCHED_WALLETS` is empty above, this cell loads the wallet set from the
stage-1 strategy registry. Skip or modify as needed.

In [171]:
import pandas as pd

if not WATCHED_WALLETS:
    if wallets_path.exists():
        _wallets_df = pd.read_parquet(wallets_path, columns=["wallet"])
        WATCHED_WALLETS = _wallets_df["wallet"].tolist()
        print(f"Loaded {len(WATCHED_WALLETS)} wallets from {wallets_path.name}")
    else:
        print("No wallet source found — set WATCHED_WALLETS manually or run stage1 first.")
else:
    print(f"Using {len(WATCHED_WALLETS)} manually specified wallets.")

Loaded 81 wallets from selected_wallets_v2.parquet


## Discover shards and derive test period

In [172]:
import pandas as pd

SHARD_PATHS: list[Path] = sorted(TRADES_DIR.glob("*.parquet"))
print(f"Found {len(SHARD_PATHS)} shards under {TRADES_DIR}")

# Derive END_DATE_TRAIN from the is_train flag (last date where is_train=True).
# Test data is everything strictly after END_DATE_TRAIN.
_sample = pd.read_parquet(SHARD_PATHS[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN: datetime.date = _sample.loc[_sample["is_train"], "dt"].max().date()
_data_end: datetime.date      = _sample["dt"].max().date()
del _sample
print(f"END_DATE_TRAIN (last train date) : {END_DATE_TRAIN}")

# Backtest always runs on test data only (strictly after END_DATE_TRAIN).
# PERIOD_START / PERIOD_END can narrow the window further, but cannot go
# earlier than the day after END_DATE_TRAIN.
#_test_start = END_DATE_TRAIN + datetime.timedelta(days=1)
period_start: datetime.date = PERIOD_START # datetime.date(2026, 2, 16) #  max(PERIOD_START, _test_start) if PERIOD_START is not None else _test_start
period_end:   datetime.date = PERIOD_END #if PERIOD_END is not None else _data_end
print(f"Backtest period : {period_start}  →  {period_end}")

Found 16 shards under ../../data/polygon_trades_processed
END_DATE_TRAIN (last train date) : 2026-04-15
Backtest period : 2026-04-16  →  2026-06-20


## Backtest engine (per-shard)

Each shard is processed independently:
1. Load only rows within the backtest period (test data only).
2. Identify trigger events (watched-wallet trades).
3. Build a per-`condition_id` opposite-outcome map (binary markets only).
4. Process triggers in time order, maintaining a copy-portfolio **position** per `(wallet, condition_id, outcome)`.
5. For each trigger, place a copy order:
   - **BUY**: `qty_ordered = min(trig_qty, STAKE_USDC / order_price)` — worst-case loss = `qty × order_price ≤ STAKE_USDC`
   - **SELL**: `qty_ordered = min(trig_qty, position, STAKE_USDC / (1 − order_price))` — worst-case loss = `qty × (1 − order_price) ≤ STAKE_USDC`. Skipped if position = 0 (no short selling).
6. Scan tape prints in time order within `FILL_HORIZON_SECONDS`. Each matching print partially fills the order: `fill_qty = min(remaining_qty, tape_qty)`.
7. One **fill** ledger row is emitted per partial fill. BUY fills increment position; SELL fills decrement it.

`fill_pnl_usdc` per fill row (all executed at `exec_price = order_price`, limit fill):
- **BUY winner**: `fill_qty × (1 − exec_price) − fill_qty × exec_price × fee`
- **BUY loser**:  `−fill_qty × exec_price × (1 + fee)`
- **SELL winner** (sold below par): `fill_qty × (exec_price − 1) − fill_qty × exec_price × fee`
- **SELL loser** (sold above par):  `fill_qty × exec_price × (1 − fee)`


In [173]:
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _build_complement_map(df: pd.DataFrame) -> dict[tuple[str, str], str]:
    """Return {(condition_id, outcome): opposite_outcome} for binary markets."""
    pairs: dict[str, set] = {}
    for cid, grp in df.groupby("condition_id", sort=False):
        pairs[cid] = set(grp["outcome"].dropna().unique())
    result: dict[tuple[str, str], str] = {}
    for cid, outcomes in pairs.items():
        if len(outcomes) == 2:
            a, b = sorted(outcomes)
            result[(cid, a)] = b
            result[(cid, b)] = a
    return result


def _simulate_shard(
    shard_path: Path,
    raw_tape_path: Path,
    watched_wallets: set[str],
    period_start: datetime.date,
    period_end: datetime.date,
    worse_price_offset: float,
    fill_horizon_seconds: float,
    stake_usdc: float,
    fee_bps: float,
    max_position_per_condition: float | None = None,
    max_position_per_wallet: float | None = None,
) -> pd.DataFrame:
    """Process one shard file and return an event ledger DataFrame.

    Triggers are read from ``shard_path`` (processed per-wallet aggregated trades),
    which supplies ``token_winner`` and ``trade_pnl``.

    The fill tape is read from ``raw_tape_path`` (raw on-chain individual fills).
    Orders are simulated chronologically against the tape so that each tape print's
    quantity can only be consumed once across all active copy orders in the shard.
    """
    TRIG_COLS = [
        "wallet", "condition_id", "outcome", "dt", "side",
        "avg_price", "total_quantity", "trade_pnl", "token_winner",
        "tx_hash",
    ]
    tdf = pd.read_parquet(shard_path, columns=TRIG_COLS)
    if tdf.empty:
        return pd.DataFrame()

    tdf["dt"] = pd.to_datetime(tdf["dt"], utc=True)
    tdf = tdf.rename(columns={"avg_price": "price", "total_quantity": "quantity"})

    date_mask = (
        (tdf["dt"].dt.date >= period_start)
        & (tdf["dt"].dt.date <= period_end)
    )
    tdf = tdf[date_mask].copy()
    if tdf.empty:
        return pd.DataFrame()

    tdf["price"] = tdf["price"].astype(float)
    tdf["quantity"] = tdf["quantity"].astype(float)

    if ONLY_BUY_TRIGGERS:
        trigger_mask = (tdf["wallet"].isin(watched_wallets)) & (tdf["side"] == "BUY")
    else:
        trigger_mask = tdf["wallet"].isin(watched_wallets)
    triggers_df = tdf[trigger_mask].copy()
    if triggers_df.empty:
        return pd.DataFrame()

    tw_map: dict[tuple[str, str], bool] = {}
    for row in tdf[["condition_id", "outcome", "token_winner"]].itertuples(index=False):
        key = (row.condition_id, row.outcome)
        if key not in tw_map and row.token_winner is not None:
            tw_map[key] = bool(row.token_winner)

    complement = _build_complement_map(tdf)
    triggers_df = triggers_df.sort_values("dt", kind="mergesort").reset_index(drop=True)

    TAPE_COLS = ["condition_id", "outcome", "block_timestamp", "side", "price", "quantity", "tx_hash", "token_winner"]
    if raw_tape_path.exists():
        rdf = pd.read_parquet(raw_tape_path, columns=TAPE_COLS)
    else:
        rdf = pd.DataFrame(columns=TAPE_COLS)

    if rdf.empty:
        ledger_rows = []
        for trig in triggers_df.itertuples(index=False):
            cid = trig.condition_id
            outcome = trig.outcome
            side = trig.side
            trig_dt = trig.dt
            trig_px = float(trig.price)
            trig_qty = float(trig.quantity)
            trig_tw = bool(trig.token_winner)
            wallet = trig.wallet
            if side == "BUY":
                order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
                qty_ordered = min(trig_qty, stake_usdc / order_px)
            else:
                order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
                qty_ordered = trig_qty
            ledger_rows.append({
                "row_type": "trigger", "order_id": len(ledger_rows) + 1,
                "wallet": wallet, "condition_id": cid, "outcome": outcome,
                "side": side, "dt": trig_dt, "tx_hash": trig.tx_hash,
                "price": trig_px, "order_price": order_px,
                "qty_ordered": qty_ordered, "qty_filled": 0.0,
                "fill_qty": None, "fill_source": None,
                "token_winner": trig_tw, "exec_price": None,
                "fill_pnl_usdc": None,
                "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
                "filled_wallet_total_position": 0.0,
                "filled_token_total_position": 0.0,
                "filled_token_by_wallet_position": 0.0,
            })
        result = pd.DataFrame(ledger_rows)
        result["shard"] = shard_path.stem
        return result

    rdf["dt"] = pd.to_datetime(rdf["block_timestamp"], unit="s", utc=True)
    rdf = rdf.drop(columns=["block_timestamp"])

    tape_start = pd.Timestamp(period_start, tz="UTC")
    tape_end = pd.Timestamp(period_end, tz="UTC") + pd.Timedelta(days=1)
    rdf = rdf[(rdf["dt"] >= tape_start) & (rdf["dt"] < tape_end)].copy()
    rdf["price"] = rdf["price"].astype(float)
    rdf["quantity"] = rdf["quantity"].astype(float)
    rdf = rdf.sort_values("dt", kind="mergesort").reset_index(drop=True)

    fee = fee_bps / 10_000.0
    horizon = pd.Timedelta(seconds=fill_horizon_seconds)
    eps = 1e-12

    ledger_rows: list[dict] = []
    order_counter = 0
    # (wallet, condition_id, outcome) → filled qty for this outcome
    positions: dict[tuple[str, str, str], float] = defaultdict(float)
    # (wallet, condition_id) → total filled qty across all outcomes of that condition
    cond_positions: dict[tuple[str, str], float] = defaultdict(float)
    # (condition_id, outcome) → total filled qty for this outcome across all wallets
    token_total_positions: dict[tuple[str, str], float] = defaultdict(float)
    # wallet → total filled qty across all conditions
    wallet_positions: dict[str, float] = defaultdict(float)

    orders: dict[int, dict] = {}
    books: dict[tuple[str, str, str], list[dict]] = defaultdict(list)

    def _append_trigger_row(
        order_id: int,
        trig,
        order_px: float,
        qty_ordered: float,
        trig_tw: bool,
        current_token_pos: float = 0.0,
        current_token_total_pos: float = 0.0,
        current_wallet_pos: float = 0.0,
    ) -> None:
        ledger_rows.append({
            "row_type": "trigger",
            "order_id": order_id,
            "wallet": trig.wallet,
            "condition_id": trig.condition_id,
            "outcome": trig.outcome,
            "side": trig.side,
            "dt": trig.dt,
            "tx_hash": trig.tx_hash,
            "price": float(trig.price),
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "qty_filled": None,
            "fill_qty": None,
            "fill_source": None,
            "token_winner": trig_tw,
            "exec_price": None,
            "fill_pnl_usdc": None,
            "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
            "filled_wallet_total_position": current_wallet_pos,
            "filled_token_total_position": current_token_total_pos,
            "filled_token_by_wallet_position": current_token_pos,
        })

    # order can match with trade on the same side and token, or opposite side and opposite token    
    def _register_order(order_id: int, order: dict, trig_tw: bool) -> None:
        cid = order["condition_id"]
        outcome = order["outcome"]
        side = order["side"]
        books[(cid, outcome, side)].append({
            "order_id": order_id,
            "fill_source": "same_token",
            "fill_tw": trig_tw,
            "opposite": False,
        })
        opp_outcome = complement.get((cid, outcome))
        if opp_outcome is None:
            return
        opp_tw = tw_map.get((cid, opp_outcome), not trig_tw)
        opp_tape_side = "SELL" if side == "BUY" else "BUY"
        books[(cid, opp_outcome, opp_tape_side)].append({
            "order_id": order_id,
            "fill_source": "opposite_token",
            "fill_tw": trig_tw,
            "opposite": True,
        })

    def _process_tape_row(row) -> None:
        key = (row.condition_id, row.outcome, row.side)
        entries = books.get(key)
        if not entries:
            return

        tape_dt = row.dt
        tape_price = float(row.price)
        tape_remaining = float(row.quantity)
        if tape_remaining <= eps:
            return

        survivors: list[dict] = []
        for entry in entries:
            order = orders.get(entry["order_id"])
            if order is None:
                continue
            if order["remaining_qty"] <= eps:
                continue
            if order["deadline"] < tape_dt:
                continue

            eff_price = float(np.clip(1.0 - tape_price, 0.001, 0.999)) if entry["opposite"] else tape_price
            eligible = eff_price <= order["order_price"] if order["side"] == "BUY" else eff_price >= order["order_price"]

            if eligible and tape_remaining > eps:
                fill_qty = min(order["remaining_qty"], tape_remaining)

                # Cap fill to remaining position headroom (per-condition and per-wallet limits).
                # This prevents overfill when a single tape print is large enough to exceed the cap.
                if order["side"] == "BUY":
                    if max_position_per_condition is not None:
                        cond_headroom = max(max_position_per_condition - cond_positions[(order["wallet"], order["condition_id"])], 0.0)
                        fill_qty = min(fill_qty, cond_headroom)
                    if max_position_per_wallet is not None:
                        wallet_headroom = max(max_position_per_wallet - wallet_positions[order["wallet"]], 0.0)
                        fill_qty = min(fill_qty, wallet_headroom)
                    if fill_qty <= eps:
                        # Position cap already reached — retire this order without consuming tape
                        order["remaining_qty"] = 0.0
                        continue

                tape_remaining -= fill_qty
                order["remaining_qty"] -= fill_qty

                if order["side"] == "BUY":
                    gross = fill_qty * (1.0 - order["order_price"]) if entry["fill_tw"] else -fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    positions[order["pos_key"]] += fill_qty
                    cond_positions[(order["wallet"], order["condition_id"])] += fill_qty
                    token_total_positions[(order["condition_id"], order["outcome"])] += fill_qty
                    wallet_positions[order["wallet"]] += fill_qty
                    new_token_pos = positions[order["pos_key"]]
                    new_token_total_pos = token_total_positions[(order["condition_id"], order["outcome"])]
                    new_wallet_pos = wallet_positions[order["wallet"]]
                else:
                    gross = fill_qty * (order["order_price"] - 1.0) if entry["fill_tw"] else fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    prev = positions[order["pos_key"]]
                    positions[order["pos_key"]] = max(prev - fill_qty, 0.0)
                    actually_reduced = prev - positions[order["pos_key"]]
                    cond_positions[(order["wallet"], order["condition_id"])] = max(
                        cond_positions[(order["wallet"], order["condition_id"])] - actually_reduced, 0.0
                    )
                    wallet_positions[order["wallet"]] = max(
                        wallet_positions[order["wallet"]] - actually_reduced, 0.0
                    )
                    token_total_positions[(order["condition_id"], order["outcome"])] = max(
                        token_total_positions[(order["condition_id"], order["outcome"])] - actually_reduced, 0.0
                    )
                    new_token_pos = positions[order["pos_key"]]
                    new_token_total_pos = token_total_positions[(order["condition_id"], order["outcome"])]
                    new_wallet_pos = wallet_positions[order["wallet"]]

                ledger_rows.append({
                    "row_type": "fill",
                    "order_id": entry["order_id"],
                    "wallet": order["wallet"],
                    "condition_id": order["condition_id"],
                    "outcome": order["outcome"],
                    "side": order["side"],
                    "dt": tape_dt,
                    "tx_hash": row.tx_hash,
                    "price": eff_price,
                    "order_price": None,
                    "qty_ordered": order["qty_ordered"],
                    "qty_filled": None,
                    "fill_qty": fill_qty,
                    "fill_source": entry["fill_source"],
                    "token_winner": entry["fill_tw"],
                    "exec_price": order["order_price"],
                    "fill_pnl_usdc": fill_pnl,
                    "wallet_trade_pnl": None,
                    "filled_wallet_total_position": new_wallet_pos,
                    "filled_token_total_position": new_token_total_pos,
                    "filled_token_by_wallet_position": new_token_pos,
                })

            if order["remaining_qty"] > eps and order["deadline"] >= tape_dt:
                survivors.append(entry)

        if survivors:
            books[key] = survivors
        else:
            books.pop(key, None)

    tape_iter = iter(rdf[["condition_id", "outcome", "dt", "side", "price", "quantity", "tx_hash", "token_winner"]].itertuples(index=False))
    current_tape = next(tape_iter, None)

    for trig in triggers_df.itertuples(index=False):
        while current_tape is not None and current_tape.dt <= trig.dt:
            _process_tape_row(current_tape)
            current_tape = next(tape_iter, None)

        cid = trig.condition_id
        outcome = trig.outcome
        side = trig.side
        trig_px = float(trig.price)
        trig_qty = float(trig.quantity)
        trig_tw = bool(trig.token_winner)
        wallet = trig.wallet
        pos_key = (wallet, cid, outcome)

        if side == "BUY":
            order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
            current_pos = positions.get(pos_key, 0.0)
            qty_ordered = min(trig_qty, stake_usdc / order_px)
        else:
            order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
            current_pos = positions.get(pos_key, 0.0)
            if current_pos <= eps:
                continue
            sell_cap = stake_usdc / max(1.0 - order_px, 1e-9)
            qty_ordered = min(trig_qty, current_pos, sell_cap)
            if qty_ordered <= eps:
                continue

        order_counter += 1
        order_id = order_counter

        order = {
            "wallet": wallet,
            "condition_id": cid,
            "outcome": outcome,
            "side": side,
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "remaining_qty": qty_ordered,
            "deadline": trig.dt + horizon,
            "pos_key": pos_key,
        }
        orders[order_id] = order

        _append_trigger_row(
            order_id,
            trig,
            order_px,
            qty_ordered,
            trig_tw,
            current_token_pos=positions.get(pos_key, 0.0),
            current_token_total_pos=token_total_positions.get((cid, outcome), 0.0),
            current_wallet_pos=wallet_positions.get(wallet, 0.0),
        )
        _register_order(order_id, order, trig_tw)

    while current_tape is not None:
        _process_tape_row(current_tape)
        current_tape = next(tape_iter, None)

    if not ledger_rows:
        return pd.DataFrame()

    result = pd.DataFrame(ledger_rows)
    result["shard"] = shard_path.stem

    filled_by_order = (
        result[result["row_type"] == "fill"]
        .groupby("order_id")["fill_qty"]
        .sum()
        .rename("_total_filled")
    )
    result = result.merge(filled_by_order, on="order_id", how="left")
    trig_mask = result["row_type"] == "trigger"
    result.loc[trig_mask, "qty_filled"] = result.loc[trig_mask, "_total_filled"].fillna(0.0)
    result = result.drop(columns=["_total_filled"])
    result["qty_filled"] = result["qty_filled"].astype(float)

    return result



## Run backtest across all shards (parallel)

In [174]:
assert WATCHED_WALLETS, "WATCHED_WALLETS is empty — set wallets in the config cell or run the wallet-load cell."

watched_set = set(WATCHED_WALLETS)
shard_results: list[pd.DataFrame] = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _simulate_shard,
            path,
            RAW_TRADES_DIR / path.name,
            watched_set,
            period_start,
            period_end,
            WORSE_PRICE_OFFSET,
            FILL_HORIZON_SECONDS,
            STAKE_USDC,
            FEE_BPS,
            MAX_POSITION_PER_CONDITION,
            MAX_POSITION_PER_WALLET,
        ): path
        for path in SHARD_PATHS
    }
    for future in as_completed(futures):
        path = futures[future]
        try:
            df = future.result()
            if df is not None and not df.empty:
                shard_results.append(df)
        except Exception as exc:
            print(f"Shard {path.name} raised an exception: {exc}")
            raise

if shard_results:
    event_ledger: pd.DataFrame = (
        pd.concat(shard_results, ignore_index=True)
        .sort_values(["dt", "shard", "order_id", "row_type"])
        .reset_index(drop=True)
    )
    _key = event_ledger[["shard", "order_id"]]
    _pairs = _key.drop_duplicates().reset_index(drop=True)
    _pairs["global_order_id"] = _pairs.index + 1
    event_ledger = event_ledger.merge(_pairs, on=["shard", "order_id"], how="left")
    event_ledger["order_id"] = event_ledger["global_order_id"]
    event_ledger = event_ledger.drop(columns=["global_order_id"])
else:
    event_ledger = pd.DataFrame(columns=[
        "row_type", "order_id", "wallet", "condition_id", "outcome",
        "side", "dt", "tx_hash", "price", "order_price",
        "qty_ordered", "qty_filled", "fill_qty",
        "fill_source", "token_winner", "exec_price", "fill_pnl_usdc",
        "wallet_trade_pnl", "shard", "filled_wallet_total_position", "filled_token_total_position", "filled_token_by_wallet_position",
    ])

triggers = event_ledger[event_ledger["row_type"] == "trigger"]
fills = event_ledger[event_ledger["row_type"] == "fill"]
filled_orders = fills["order_id"].nunique()

print(f"Trigger events    : {len(triggers):>7,}")
print(f"Fill rows         : {len(fills):>7,}")
print(f"Orders with fills : {filled_orders:>7,}")
if len(triggers) > 0:
    print(f"Order fill rate   : {filled_orders/len(triggers)*100:.1f}%")
else:
    print("No trigger events found — check WATCHED_WALLETS and the period.")



Trigger events    :  54,075
Fill rows         :  14,529
Orders with fills :   9,258
Order fill rate   : 17.1%


## Summary statistics

In [175]:
fee = FEE_BPS / 10_000.0

if not fills.empty:
    filled_copy_pnl    = fills["fill_pnl_usdc"].sum()
    total_qty_filled   = fills["fill_qty"].sum()
    total_notional     = (fills["fill_qty"] * fills["exec_price"]).sum()
    orders_with_fills  = fills["order_id"].nunique()
    partial_orders     = (fills.groupby("order_id").size() > 1).sum()
    win_rate           = fills.groupby("order_id")["token_winner"].first().mean()
    avg_exec_price     = fills["exec_price"].mean()
    fill_src_counts    = fills["fill_source"].value_counts()

    # Partial-fill statistics
    trig_qty = triggers.set_index("order_id")["qty_ordered"]
    fill_qty = triggers.set_index("order_id")["qty_filled"]
    fill_pct = (fill_qty / trig_qty.clip(lower=1e-12) * 100).replace([float('inf'), float('nan')], 0)

    print(f"=== Copy-strategy performance ===")
    print(f"  Orders triggered    : {len(triggers):>7,}")
    print(f"  Orders with fills   : {orders_with_fills:>7,}  ({orders_with_fills/len(triggers)*100:.1f}%)")
    print(f"  Orders multi-fill   : {partial_orders:>7,}  ({partial_orders/max(orders_with_fills,1)*100:.1f}% of filled)")
    print(f"  Avg fill %          : {fill_pct[fill_pct>0].mean():>7.1f}%")
    print(f"  Total qty filled    : {total_qty_filled:>10,.2f} tokens")
    print(f"  Total notional      : {total_notional:>10,.2f} USDC")
    print(f"  Net PnL (USDC)      : {filled_copy_pnl:>10,.2f}")
    print(f"  Net ROI on notional : {filled_copy_pnl/total_notional*100:>8.2f}%")
    print(f"  Win rate (by order) : {win_rate*100:>8.2f}%")
    print(f"  Avg exec price      : {avg_exec_price:>8.4f}")
    print(f"\n  Fill source breakdown:")
    for src, cnt in fill_src_counts.items():
        print(f"    {src:<20s}: {cnt:>6,}  ({cnt/len(fills)*100:.1f}%)")
else:
    print("No fills — nothing to summarise.")

# Watched-wallet cohort summary
wallet_pnl = triggers["wallet_trade_pnl"].sum()
print(f"\n=== Watched-wallet cohort ===")
print(f"  Total trades        : {len(triggers):>7,}")
print(f"  Total trade PnL     : {wallet_pnl:>10,.2f} USDC")


=== Copy-strategy performance ===
  Orders triggered    :  54,075
  Orders with fills   :   9,258  (17.1%)
  Orders multi-fill   :   2,960  (32.0% of filled)
  Avg fill %          :    86.4%
  Total qty filled    : 122,442.01 tokens
  Total notional      :  50,510.11 USDC
  Net PnL (USDC)      :     763.27
  Net ROI on notional :     1.51%
  Win rate (by order) :    44.30%
  Avg exec price      :   0.4371

  Fill source breakdown:
    same_token          : 10,547  (72.6%)
    opposite_token      :  3,982  (27.4%)

=== Watched-wallet cohort ===
  Total trades        :  54,075
  Total trade PnL     : 431,202.39 USDC


## Event ledger preview

In [176]:
event_ledger[
    (event_ledger['row_type'] == 'fill')
    & (event_ledger['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20')
    & (event_ledger['wallet'] == '0x0b219cf3d297991b58361dbebdbaa91e56b8deb6')
    ].head(500)

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard


In [177]:
pd.set_option('display.max_rows', 100)

In [178]:
display_cols = [
    "row_type", "order_id", "wallet", "condition_id", "outcome", "side",
    "dt", "tx_hash", "price", "order_price", "exec_price",
    "qty_ordered", "qty_filled", "fill_qty",
    "fill_source", "token_winner", "fill_pnl_usdc", "filled_wallet_total_position", "filled_token_total_position", "filled_token_by_wallet_position", "wallet_trade_pnl",
]
available = [c for c in display_cols if c in event_ledger.columns]

# Show a few trigger+fill pairs
sample_orders = event_ledger["order_id"].unique()[:5]
event_ledger[
    event_ledger["order_id"].isin(sample_orders)
][available].sort_values(["order_id", "row_type"])


,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,fill_pnl_usdc,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,wallet_trade_pnl
0,trigger,1,0x74cbe13dba27a6a16805e9e7142ee68aa09cae6d,0x6b09d2bd85728308faa51a74f78b1f27e798a4ff0751c70ac112d5dc11825832,No,BUY,2026-04-16 00:00:18+00:00,0x540db9a920f5f357a16b5fa60298f56cf43da4830ec2495a969c4622596dc1b7,0.002,0.002,...,168.380000,0.000000,NaN,None,False,NaN,0.000000,0.000000,0.000000,-0.33676
1,trigger,2,0xc468b3b8564017fcf2bf9ede2608a0ea24b1009d,0x6f94ad47610b2f7540c94382082fb6f5fc1a5e648d63bac79924fe695935642c,No,BUY,2026-04-16 00:07:16+00:00,0x0ac0827923a47fce87645b4c2b79568dc0de10e54028ac600c7b56dcb1d5e8b8,0.953,0.953,...,10.493179,0.000000,NaN,None,True,NaN,0.000000,0.000000,0.000000,2.35000
2,trigger,3,0x3c760e45728e0cb4b68c52980a070314b95fd4ee,0x11f55c831e6697fcd2624209c701f77436e08008695c743c47dfe94d26636069,No,BUY,2026-04-16 00:24:48+00:00,0x198e1426e73b0938069cc16cf12d2251610f9ff2cdf8c121ba67c9bbda497a1d,0.700,0.700,...,14.285714,0.000000,NaN,None,True,NaN,0.000000,0.000000,0.000000,15.00000
3,trigger,4,0x9f16feb66b8a4355528fd9eb1f6fceb989337013,0xfb2031685ad36fa6a923d072a3f0e72b10847528eadcc47121b81737a996459d,Yes,BUY,2026-04-16 00:33:54+00:00,0xe385e6ec22cc630ae519ab7753bb7218f0f509cacb766aee1996f35540fda922,0.011,0.011,...,9.280000,0.000000,NaN,None,False,NaN,0.000000,0.000000,0.000000,-0.10208
5,fill,5,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,Yes,BUY,2026-04-16 00:38:16+00:00,0x70f75228c3bd4c3019cbfdade117c968ef57431bd50766855ae7c8665744c64f,0.683,NaN,...,14.641288,NaN,5.210000,same_token,True,1.648012,5.210000,5.210000,5.210000,NaN
6,fill,5,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,Yes,BUY,2026-04-16 00:38:16+00:00,0x70f75228c3bd4c3019cbfdade117c968ef57431bd50766855ae7c8665744c64f,0.683,NaN,...,14.641288,NaN,7.000000,same_token,True,2.214219,12.210000,12.210000,12.210000,NaN
7,fill,5,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,Yes,BUY,2026-04-16 00:38:16+00:00,0x70f75228c3bd4c3019cbfdade117c968ef57431bd50766855ae7c8665744c64f,0.682,NaN,...,14.641288,NaN,2.431288,same_token,True,0.769058,14.641288,14.641288,14.641288,NaN
4,trigger,5,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,Yes,BUY,2026-04-16 00:36:40+00:00,0xd11db67c7cbf13620953aa97959d87dfeeb4f73c03be530d6bceab6e0cfb17e5,0.683,0.683,...,14.641288,14.641288,NaN,None,True,NaN,0.000000,0.000000,0.000000,158.50000


In [179]:
event_ledger[event_ledger['side'] == 'BUY'].groupby('wallet').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    buy_count=('side', 'count')
)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,buy_count
wallet,,,
0x0711e162e05349de3d87626dea4285d08537f03c,95.041502,3316.672367,3384
0x081688cdfa52f2edcc3c2e56c02399314d99377a,-59.732580,-203.305007,153
0x0cb10c40b0776e9ee8cef970af85724654dda76c,-43.901982,6618.848316,1379
0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,125.633199,1771.606019,196
0x17c6b26eaeb2a6a2ef768ae0e27c9591e2a08aca,0.422105,217.499990,2
0x1c144e30f405a25f991cbd8baa15d40599090869,-159.937771,-585.227345,751
0x1c4cd350bce3cb791daf88ce2034de9b03318d85,229.770150,7985.280464,608
0x22dbd51e1a4e10fff072fa24300238c64033190f,-141.468184,14404.384522,2741
0x2d4bf8f846bf68f43b9157bf30810d334ac6ca7a,25.113424,78224.087858,165


In [180]:
event_ledger[event_ledger['side'] == 'BUY'].groupby('condition_id').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    buy_count=('side', 'count')
).sort_values('fill_pnl_usdc_sum', ascending=False).head(10)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,buy_count
condition_id,,,
0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,795.979509,374143.916590,966
0xaa145cf5714455f546afe53037b8712df749ba96c04820c2d41f37f877f29697,474.817625,25741.554082,409
0x471e2fa06bd8975b77be49a36781bc41286d57bffa84de55173f4eacdbd52b28,376.111008,4043.503616,231
0x5341e90b72330a8eb96cb685dcffe51ad41783cc12e91b6858b9bd996b3d5b63,344.605379,6825.148953,233
0x16608c4fdd7cb41292f6d42c1c02b43ac22593f1c198a104847a3479848d7abe,146.669210,11956.754236,237
0x2135ffcb43ba9103bb6acf7116d2d5aa98bef6d9eb3dc9c85ea00cb79513f3ec,125.187023,8724.202571,366
0x52bda1b8d68c469eb3176f2052f4da7cacc65a814c51f53294637d124af6afde,123.986885,2425.256831,89
0xe546672750517f62c45a5a00067481981e62b9c20fa8220203232c9dc8fd2093,123.662622,5812.989065,152
0xd8423c8dc3174c6839b0cf667ce1a1364f85325739409e0f187562227cae44cc,107.781268,21218.062762,22


In [181]:
(fills[fills['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20'][['dt', 'side', 'outcome',
                                                                                                        'price', 'fill_qty', 'exec_price', 
                                                                                                        # 'filled_wallet_total_position',
                                                                                                          'filled_token_total_position', 
                                                                                                          'filled_token_by_wallet_position', 
                                                                                                          'fill_pnl_usdc',
                                                                                                          'order_id','wallet', 'tx_hash']]
 .sort_values('dt')
 .head(10))

,dt,side,outcome,price,fill_qty,exec_price,filled_token_total_position,filled_token_by_wallet_position,fill_pnl_usdc,order_id,wallet,tx_hash


In [182]:
sample_filled_orders = fills["order_id"].unique()[:50]
event_ledger[
    (event_ledger['side'] == 'SELL') &
    (event_ledger["order_id"].isin(sample_filled_orders))
].sort_values(["order_id", "dt"])

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard
105,trigger,67,0x39d0f1dca6fb7e5514858c1a337724a426764fe8,0x1161c3b9969ad118659e604b2442c858bca0f76d1564e277178e5cf628dac974,Yes,SELL,2026-04-16 03:58:08+00:00,0x7225adeaf9960c458ff0bdc097c61e67d7d9d1d40d5f29fe900e8bed88c47459,0.25,0.25,...,NaN,None,True,NaN,NaN,-130.5000,43.826667,43.826667,43.826667,1
106,fill,67,0x39d0f1dca6fb7e5514858c1a337724a426764fe8,0x1161c3b9969ad118659e604b2442c858bca0f76d1564e277178e5cf628dac974,Yes,SELL,2026-04-16 03:58:14+00:00,0xd309ed140901b2ecde6b923c5c35bb8286dfd4b9215fa009084ac4867185091f,0.26,NaN,...,13.333333,same_token,True,0.25,-10.003333,NaN,30.493333,30.493333,30.493333,1
107,trigger,68,0x39d0f1dca6fb7e5514858c1a337724a426764fe8,0x1161c3b9969ad118659e604b2442c858bca0f76d1564e277178e5cf628dac974,Yes,SELL,2026-04-16 03:58:30+00:00,0xc75aa19926d873a545d1ec9c9e0c9ab4dae398e2d35bb159032d7c607de7fafc,0.26,0.26,...,NaN,None,True,NaN,NaN,-9.3018,30.493333,30.493333,30.493333,1
108,fill,68,0x39d0f1dca6fb7e5514858c1a337724a426764fe8,0x1161c3b9969ad118659e604b2442c858bca0f76d1564e277178e5cf628dac974,Yes,SELL,2026-04-16 03:59:00+00:00,0xfb0b00dc7cddbb8752640e28cb82cbe73dbf9346a3f35cbf87f06237ee09cb7d,0.26,NaN,...,5.000000,same_token,True,0.26,-3.701300,NaN,25.493333,25.493333,25.493333,1
116,trigger,74,0x39d0f1dca6fb7e5514858c1a337724a426764fe8,0x1161c3b9969ad118659e604b2442c858bca0f76d1564e277178e5cf628dac974,Yes,SELL,2026-04-16 04:09:10+00:00,0x8d34fce27aef613c37e3ecfad4f887aeb78adc065bd063bcca93a79084aa98e7,0.25,0.25,...,NaN,None,True,NaN,NaN,-3.0900,25.493333,25.493333,25.493333,1
117,fill,74,0x39d0f1dca6fb7e5514858c1a337724a426764fe8,0x1161c3b9969ad118659e604b2442c858bca0f76d1564e277178e5cf628dac974,Yes,SELL,2026-04-16 04:09:52+00:00,0x54fbc0d88859c212dd7cd22969858543ee3aab0aee09491c8bded1e54795243e,0.25,NaN,...,4.120000,same_token,True,0.25,-3.091030,NaN,21.373333,21.373333,21.373333,1
118,trigger,75,0x39d0f1dca6fb7e5514858c1a337724a426764fe8,0x1161c3b9969ad118659e604b2442c858bca0f76d1564e277178e5cf628dac974,Yes,SELL,2026-04-16 04:09:52+00:00,0x54fbc0d88859c212dd7cd22969858543ee3aab0aee09491c8bded1e54795243e,0.25,0.25,...,NaN,None,True,NaN,NaN,-64.3800,21.373333,21.373333,21.373333,1
122,fill,75,0x39d0f1dca6fb7e5514858c1a337724a426764fe8,0x1161c3b9969ad118659e604b2442c858bca0f76d1564e277178e5cf628dac974,Yes,SELL,2026-04-16 04:13:34+00:00,0xcfb31782796acb62fd9208fe746c29cf6a9d6bbb975a9b42b7bd2301f4b8b5d3,0.25,NaN,...,0.850000,same_token,True,0.25,-0.637712,NaN,20.523333,20.523333,20.523333,1
123,fill,75,0x39d0f1dca6fb7e5514858c1a337724a426764fe8,0x1161c3b9969ad118659e604b2442c858bca0f76d1564e277178e5cf628dac974,Yes,SELL,2026-04-16 04:13:56+00:00,0x49e4c86b3d04772eed0d2d3b2c7a9cf47d96193f78eae62b60010cd5991154c1,0.25,NaN,...,5.000000,same_token,True,0.25,-3.751250,NaN,15.523333,15.523333,15.523333,1
124,fill,75,0x39d0f1dca6fb7e5514858c1a337724a426764fe8,0x1161c3b9969ad118659e604b2442c858bca0f76d1564e277178e5cf628dac974,Yes,SELL,2026-04-16 04:14:16+00:00,0x399d19bcf59f776bff08553b3411abccfca606b3cf7fa6d2f1c018b0b0cef042,0.25,NaN,...,5.000000,same_token,True,0.25,-3.751250,NaN,10.523333,10.523333,10.523333,1


## Build PnL time series

In [183]:
def resample_hourly_series(df: pd.DataFrame, dt_col: str, pnl_col: str) -> pd.DataFrame:
    """Resample a PnL column to 1-h buckets and compute cumulative sum."""
    if df.empty or pnl_col not in df.columns:
        return pd.DataFrame(columns=["trade_dt", "net_pnl_usdc", "cum_net_pnl_usdc"])
    hourly = (
        df.assign(trade_dt=df[dt_col].dt.floor("1h"))
        .groupby("trade_dt", as_index=False)[pnl_col]
        .sum()
        .rename(columns={pnl_col: "net_pnl_usdc"})
        .sort_values("trade_dt")
        .reset_index(drop=True)
    )
    hourly["cum_net_pnl_usdc"] = hourly["net_pnl_usdc"].cumsum()
    return hourly


def with_zero_anchor(hourly: pd.DataFrame) -> pd.DataFrame:
    """Prepend a zero anchor one hour before the first bucket."""
    if hourly.empty:
        return hourly
    anchor = pd.DataFrame({
        "trade_dt": [hourly["trade_dt"].min() - pd.Timedelta(hours=1)],
        "net_pnl_usdc": [0.0],
        "cum_net_pnl_usdc": [0.0],
    })
    return pd.concat([anchor, hourly], ignore_index=True)


# Wallet cohort PnL: from trigger rows (their actual trade_pnl)
wallet_hourly = resample_hourly_series(triggers, "dt", "wallet_trade_pnl")

# Copy-strategy PnL: from fill rows
copy_hourly = resample_hourly_series(fills, "dt", "fill_pnl_usdc")

print(f"Wallet cohort hourly buckets : {len(wallet_hourly)}")
print(f"Copy strategy hourly buckets : {len(copy_hourly)}")

Wallet cohort hourly buckets : 1476
Copy strategy hourly buckets : 1126


In [184]:
wallet_hourly.head()

,trade_dt,net_pnl_usdc,cum_net_pnl_usdc
0,2026-04-16 00:00:00+00:00,175.411160,175.411160
1,2026-04-16 01:00:00+00:00,10.399499,185.810659
2,2026-04-16 02:00:00+00:00,-1281.976206,-1096.165547
3,2026-04-16 03:00:00+00:00,554.223766,-541.941781
4,2026-04-16 04:00:00+00:00,-177.630855,-719.572636


## Cumulative PnL chart

In [185]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = go.Figure()

# ── Wallet cohort trace ───────────────────────────────────────────────────────
if not wallet_hourly.empty:
    wh = with_zero_anchor(wallet_hourly)
    fig.add_trace(go.Scatter(
        x=wh["trade_dt"],
        y=wh["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="dot", width=2),
        opacity=0.7,
        name="Watched wallets (raw PnL)",
        hovertemplate=(
            "<b>Watched wallets</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

# ── Copy-strategy trace ───────────────────────────────────────────────────────
if not copy_hourly.empty:
    ch = with_zero_anchor(copy_hourly)
    fig.add_trace(go.Scatter(
        x=ch["trade_dt"],
        y=ch["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="solid", width=2),
        name="Copy strategy (filled)",
        hovertemplate=(
            "<b>Copy strategy</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

fig.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5)

fig.update_layout(
    template="plotly_white",
    height=480,
    title=dict(
        text=(
            f"Copy-trade backtest — cumulative PnL (1h) | "
            f"{period_start} → {period_end} | "
            f"{len(WATCHED_WALLETS)} wallets | "
            f"worse_price={WORSE_PRICE_OFFSET:.2f} | "
            f"horizon={int(FILL_HORIZON_SECONDS)}s"
        ),
        font=dict(size=13),
    ),
    xaxis_title="Time (1h buckets)",
    yaxis_title="Cumulative net PnL (USDC)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

fig.show()

## Diagnostics

In [186]:
# Fill rate by side (order-level)
if not triggers.empty:
    trig_by_side = triggers.groupby("side").size().rename("triggers")
    fill_by_side = fills.groupby("side")["order_id"].nunique().rename("orders_filled")
    fill_rows_by_side = fills.groupby("side").size().rename("fill_rows")
    side_summary = pd.concat([trig_by_side, fill_by_side, fill_rows_by_side], axis=1).fillna(0).astype(int)
    side_summary["fill_rate"] = (side_summary["orders_filled"] / side_summary["triggers"] * 100).round(1)
    display(side_summary)


,triggers,orders_filled,fill_rows,fill_rate
side,,,,
BUY,49014,5973,9689,12.2
SELL,5061,3285,4840,64.9


In [187]:
# Fill source breakdown by side
if not fills.empty:
    display(
        fills.groupby(["side", "fill_source"]).size()
        .rename("count")
        .reset_index()
        .sort_values(["side", "fill_source"])
    )

,side,fill_source,count
0,BUY,opposite_token,2500
1,BUY,same_token,7189
2,SELL,opposite_token,1482
3,SELL,same_token,3358


In [188]:
df = event_ledger.assign(
    is_trigger=event_ledger["row_type"].eq("trigger"),
    is_fill=event_ledger["row_type"].eq("fill"),
)

# --- wallet-level stats (row-based) ---
wallet_stats = (
    df.groupby("wallet")
    .agg(
        total_triggers=("is_trigger", "sum"),
        total_fills=("is_fill", "sum"),
        total_fill_pnl=("fill_pnl_usdc", "sum"),
        total_trade_pnl=("wallet_trade_pnl", "sum"),
        total_pnl=("fill_pnl_usdc", "sum"),
    )
)

# --- order-level logic (trigger -> fill) ---
order_stats = (
    df.groupby(["wallet", "order_id"])
    .agg(
        has_trigger=("is_trigger", "any"),
        has_fill=("is_fill", "any"),
    )
    .assign(trigger_with_fill=lambda x: x["has_trigger"] & x["has_fill"])
    .groupby("wallet")
    .agg(triggers_with_fill=("trigger_with_fill", "sum"))
)

# --- combine ---
result = wallet_stats.join(order_stats)

# --- derived metric ---
result["fill_ratio"] = (
    result["triggers_with_fill"] / result["total_triggers"]
).fillna(0)

result.sort_values("total_pnl", ascending=False).head()

,total_triggers,total_fills,total_fill_pnl,total_trade_pnl,total_pnl,triggers_with_fill,fill_ratio
wallet,,,,,,,
0xa0bca9bdd8540da95060ed1fafb78aa03835d428,1865,633,269.080804,-4031.427432,269.080804,440,0.235925
0x6df6e2a9ba1e8d7609daada0a83138817f4a8458,125,93,225.159092,11527.738231,225.159092,61,0.488000
0xe732156a2d84cdfb4de831d3f11a22899e49898f,778,467,191.939657,8981.247434,191.939657,294,0.377892
0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292,816,499,181.823713,91822.732191,181.823713,284,0.348039
0x74cbe13dba27a6a16805e9e7142ee68aa09cae6d,333,180,179.716991,723.280874,179.716991,125,0.375375


In [189]:
len(df)

68604

In [190]:
# get diff between trigger and fill timestamp per order_id
# without lambda, using groupby + agg + merge
trigger_df = df[df['row_type'] == 'trigger'].groupby('order_id')['dt'].min().reset_index()
fill_df = df[df['row_type'] == 'fill'].groupby('order_id')['dt'].min().reset_index()

merged_df = pd.merge(trigger_df, fill_df, on='order_id', how='outer', suffixes=('_trigger', '_fill'))
merged_df['diff_dt'] = merged_df['dt_fill'] - merged_df['dt_trigger']

In [191]:
result[result['total_fill_pnl'] < 0].sort_values("total_fill_pnl").index.tolist()

['0x39d0f1dca6fb7e5514858c1a337724a426764fe8',
 '0xfe9339e4eca140c1a16f94a52b5cde4d29768d0e',
 '0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22',
 '0x969fae0a3a93778adc42178f72c612ed8c4e4d55',
 '0x6640bd87f6e4b6e8d62457448bd1b3a4711a2202',
 '0x66c1a6fe836ff555ca32848646acedbbe93bfa3f',
 '0xc468b3b8564017fcf2bf9ede2608a0ea24b1009d',
 '0x41583f2efc720b8e2682750fffb67f2806fece9f',
 '0x1c144e30f405a25f991cbd8baa15d40599090869',
 '0x081688cdfa52f2edcc3c2e56c02399314d99377a',
 '0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f',
 '0x596a78ff6de24b12fd39f827aecf93975e2f7a0f',
 '0x7656ed7f597a0a61cd307591db198a42b2a7194b',
 '0x7fa1507e1803272c26d91fb263c507b995d3deeb',
 '0x99984e22205053950eb25453779267bcc1aee858',
 '0xcc3e87226912d35893b678eaa8a39d57b6fe37ef',
 '0x373a949d617e60cbb25ca6df3f68018d573bf4c1',
 '0xb94f0ea2ddde1779122834ec827b56afe82804d8',
 '0x82178ad548a5fd769583a1c361b34b966ae139b1',
 '0x9f16feb66b8a4355528fd9eb1f6fceb989337013',
 '0x92d0cb81e6c891b835c8e0272e8c323095bd269e',
 '0x566d32ba7

In [192]:
# Per-wallet PnL breakdown
if not fills.empty:
    wallet_pnl_df = (
        fills.groupby("wallet")
        .agg(
            filled_orders=("order_id", "nunique"),
            fill_rows=("order_id", "count"),
            net_pnl_usdc=("fill_pnl_usdc", "sum"),
            total_qty=("fill_qty", "sum"),
            win_rate=("token_winner", lambda s: fills.loc[s.index].groupby("order_id")["token_winner"].first().mean()),
        )
        .assign(win_rate=lambda d: (d["win_rate"] * 100).round(1))
        .sort_values("net_pnl_usdc", ascending=False)
        .reset_index()
    )
    display(wallet_pnl_df.sort_values("net_pnl_usdc", ascending=False, key=abs))


,wallet,filled_orders,fill_rows,net_pnl_usdc,total_qty,win_rate
57,0x39d0f1dca6fb7e5514858c1a337724a426764fe8,1018,1684,-363.710055,13462.574023,33.0
0,0xa0bca9bdd8540da95060ed1fafb78aa03835d428,440,633,269.080804,5346.009359,49.5
56,0xfe9339e4eca140c1a16f94a52b5cde4d29768d0e,279,398,-267.708063,2146.611737,34.1
55,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,384,528,-259.030601,4772.963219,58.6
1,0x6df6e2a9ba1e8d7609daada0a83138817f4a8458,61,93,225.159092,1026.283362,75.4
2,0xe732156a2d84cdfb4de831d3f11a22899e49898f,294,467,191.939657,4479.804390,46.3
3,0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292,284,499,181.823713,4301.240251,52.5
54,0x969fae0a3a93778adc42178f72c612ed8c4e4d55,260,354,-180.603185,3222.579475,64.2
4,0x74cbe13dba27a6a16805e9e7142ee68aa09cae6d,125,180,179.716991,2068.805475,28.0
5,0xca12d28728c46d3484395243f5f842f2c321ea03,112,149,177.153330,1755.268253,52.7


In [193]:
# Orders that were NOT filled at all
filled_order_ids = set(fills["order_id"].unique()) if not fills.empty else set()
unfilled_triggers = triggers[~triggers["order_id"].isin(filled_order_ids)]
print(f"Unfilled orders    : {len(unfilled_triggers):,} ({len(unfilled_triggers)/max(len(triggers),1)*100:.1f}% of all triggers)")

# Partially filled orders (filled but qty_filled < qty_ordered)
if not fills.empty:
    filled_trig = triggers[triggers["order_id"].isin(filled_order_ids)].copy()
    partial_mask = filled_trig["qty_filled"] < filled_trig["qty_ordered"] - 1e-9
    partial_fills = filled_trig[partial_mask]
    print(f"Partially filled   : {len(partial_fills):,} ({len(partial_fills)/max(len(filled_trig),1)*100:.1f}% of filled orders)")
    print(f"Fully filled       : {len(filled_trig)-len(partial_fills):,}")

if not unfilled_triggers.empty:
    print("\nUnfilled breakdown by side:")
    display(unfilled_triggers.groupby("side").size().rename("count").reset_index())


Unfilled orders    : 44,817 (82.9% of all triggers)
Partially filled   : 2,184 (23.6% of filled orders)
Fully filled       : 7,074

Unfilled breakdown by side:


,side,count
0,BUY,43041
1,SELL,1776


In [194]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
fills[fills['order_id'] == 34]

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard


In [195]:
fills.groupby("wallet")["fill_pnl_usdc"].sum().sort_values(ascending=False, key=abs).head(100)

wallet
0x39d0f1dca6fb7e5514858c1a337724a426764fe8   -363.710055
0xa0bca9bdd8540da95060ed1fafb78aa03835d428    269.080804
0xfe9339e4eca140c1a16f94a52b5cde4d29768d0e   -267.708063
0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22   -259.030601
0x6df6e2a9ba1e8d7609daada0a83138817f4a8458    225.159092
0xe732156a2d84cdfb4de831d3f11a22899e49898f    191.939657
0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292    181.823713
0x969fae0a3a93778adc42178f72c612ed8c4e4d55   -180.603185
0x74cbe13dba27a6a16805e9e7142ee68aa09cae6d    179.716991
0xca12d28728c46d3484395243f5f842f2c321ea03    177.153330
0x1c4cd350bce3cb791daf88ce2034de9b03318d85    165.971176
0x3b4484b6c8cbfdaa383ba337ab3f0d71055e264e    125.785182
0x0dedae6a02ea2ff8018ba5f277632919ed1c9025    106.441293
0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62    105.369035
0x9bd74f5f970fe3d3c5e747fabcdc025fa47796d3    102.242396
0x3c760e45728e0cb4b68c52980a070314b95fd4ee     99.454458
0x6640bd87f6e4b6e8d62457448bd1b3a4711a2202    -96.671831
0x9b2fcb79bdb8a6a6174da3

In [196]:
trigger_wallets = triggers[["order_id", "wallet"]].set_index("order_id")["wallet"]

fills_with_wallet = fills.merge(trigger_wallets, on="order_id", how="left", suffixes=("", "_trigger"))

fills_with_wallet['notional'] = fills_with_wallet['fill_qty'] * fills_with_wallet['exec_price']

fills_with_wallet.groupby(["wallet", "condition_id"]).agg(
    pnl=("fill_pnl_usdc", "sum"),
    orders_filled=("order_id", "nunique")
    ).sort_values(['wallet', "pnl"], ascending=[True, False]).head(1)

,,pnl,orders_filled
wallet,condition_id,,
0x0711e162e05349de3d87626dea4285d08537f03c,0xa7edf95bf27a464dd6126e977eaffccb1afacdc0e4f9506cc97b1d6d28c7bc6f,33.937731,5


In [197]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard,wallet_trigger,notional
0,fill,5,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,Yes,BUY,2026-04-16 00:38:16+00:00,0x70f75228c3bd4c3019cbfdade117c968ef57431bd50766855ae7c8665744c64f,0.683,NaN,...,True,0.683000,1.648012,NaN,5.210000,5.210000,5.210000,c,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,3.558430
1,fill,5,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,Yes,BUY,2026-04-16 00:38:16+00:00,0x70f75228c3bd4c3019cbfdade117c968ef57431bd50766855ae7c8665744c64f,0.683,NaN,...,True,0.683000,2.214219,NaN,12.210000,12.210000,12.210000,c,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,4.781000
2,fill,5,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,Yes,BUY,2026-04-16 00:38:16+00:00,0x70f75228c3bd4c3019cbfdade117c968ef57431bd50766855ae7c8665744c64f,0.682,NaN,...,True,0.683000,0.769058,NaN,14.641288,14.641288,14.641288,c,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,1.660570
3,fill,6,0xff052a89d46a289a32283dfa7815ba23da49fad7,0x0b405dfa1d9ea2a8f0e97b1052ef2b0000f67bddff0242ff4ec406c6e3df5cfa,Yes,BUY,2026-04-16 01:36:34+00:00,0x19e0a85d77e95da819774e6914165231e8c7b45f6f51300f8223ec3aab05928a,0.004,NaN,...,False,0.004471,-0.062661,NaN,14.000000,14.000000,14.000000,0,0xff052a89d46a289a32283dfa7815ba23da49fad7,0.062599
4,fill,8,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,No,BUY,2026-04-16 01:52:00+00:00,0x55e617b3156c9a4f09baeab2d6c7e7b7725ec31acc6bb0418a05f930f0954072,0.850,NaN,...,True,0.850000,0.524657,NaN,3.517646,3.517646,3.517646,6,0x22dbd51e1a4e10fff072fa24300238c64033190f,2.989999


In [198]:
wallet_pnls = (
    fills_with_wallet
    .groupby(["wallet", "condition_id"])
    .agg(
        pnl=("fill_pnl_usdc", "sum"),
        orders_filled=("order_id", "nunique"),
        notional=("notional", "sum"),
    )
    .groupby("wallet")
    .agg(
        pnl=("pnl", "sum"),
        notional=("notional", "sum"),
        unique_conditions=("pnl", "size"),  # ✅ count rows
        total_orders_filled=("orders_filled", "sum"),
    )
    .sort_values("pnl", ascending=False)
)

In [199]:
wallet_pnls.head()

,pnl,notional,unique_conditions,total_orders_filled
wallet,,,,
0xa0bca9bdd8540da95060ed1fafb78aa03835d428,269.080804,2149.592307,90,440
0x6df6e2a9ba1e8d7609daada0a83138817f4a8458,225.159092,431.996729,12,61
0xe732156a2d84cdfb4de831d3f11a22899e49898f,191.939657,2112.781369,72,294
0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292,181.823713,1851.728120,35,284
0x74cbe13dba27a6a16805e9e7142ee68aa09cae6d,179.716991,380.518833,24,125


In [200]:
MAX_WALLET_EXPOSURE = 1000.0  # USDC
wallet_pnls['pnl_limited'] = np.where(
    wallet_pnls['notional'] <= MAX_WALLET_EXPOSURE,
    wallet_pnls['pnl'],
    wallet_pnls['pnl'] * MAX_WALLET_EXPOSURE / wallet_pnls['notional']
)

In [201]:
wallet_pnls.head(10)

,pnl,notional,unique_conditions,total_orders_filled,pnl_limited
wallet,,,,,
0xa0bca9bdd8540da95060ed1fafb78aa03835d428,269.080804,2149.592307,90,440,125.177599
0x6df6e2a9ba1e8d7609daada0a83138817f4a8458,225.159092,431.996729,12,61,225.159092
0xe732156a2d84cdfb4de831d3f11a22899e49898f,191.939657,2112.781369,72,294,90.846909
0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292,181.823713,1851.728120,35,284,98.191365
0x74cbe13dba27a6a16805e9e7142ee68aa09cae6d,179.716991,380.518833,24,125,179.716991
0xca12d28728c46d3484395243f5f842f2c321ea03,177.153330,628.688352,54,112,177.153330
0x1c4cd350bce3cb791daf88ce2034de9b03318d85,165.971176,1342.053961,42,277,123.669525
0x3b4484b6c8cbfdaa383ba337ab3f0d71055e264e,125.785182,218.995992,22,44,125.785182
0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,106.441293,144.206406,17,26,106.441293


In [202]:
result = (
    fills_with_wallet
    # 1️⃣ PnL per market (condition) per wallet
    .groupby(["wallet", "condition_id"])["fill_pnl_usdc"]
    .sum()
    
    # 2️⃣ Then per wallet
    .groupby("wallet")
    .agg(
        unique_conditions="count",   # number of markets traded
        median_market_pnl="median",  # median PnL across markets
    )
    .sort_values("unique_conditions", ascending=False)
    .head(20)
)

In [203]:
result.sort_values("median_market_pnl", ascending=False).head(100)

,unique_conditions,median_market_pnl
wallet,,
0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8,60,0.805136
0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,58,0.332284
0x22dbd51e1a4e10fff072fa24300238c64033190f,79,0.317296
0xc468b3b8564017fcf2bf9ede2608a0ea24b1009d,67,0.300297
0x1c144e30f405a25f991cbd8baa15d40599090869,53,0.225031
0xa0bca9bdd8540da95060ed1fafb78aa03835d428,90,0.214878
0x3c760e45728e0cb4b68c52980a070314b95fd4ee,81,0.153493
0xca12d28728c46d3484395243f5f842f2c321ea03,54,0.138154
0xe732156a2d84cdfb4de831d3f11a22899e49898f,72,0.105695


In [204]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_wallet_total_position,filled_token_total_position,filled_token_by_wallet_position,shard,wallet_trigger,notional
0,fill,5,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,Yes,BUY,2026-04-16 00:38:16+00:00,0x70f75228c3bd4c3019cbfdade117c968ef57431bd50766855ae7c8665744c64f,0.683,NaN,...,True,0.683000,1.648012,NaN,5.210000,5.210000,5.210000,c,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,3.558430
1,fill,5,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,Yes,BUY,2026-04-16 00:38:16+00:00,0x70f75228c3bd4c3019cbfdade117c968ef57431bd50766855ae7c8665744c64f,0.683,NaN,...,True,0.683000,2.214219,NaN,12.210000,12.210000,12.210000,c,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,4.781000
2,fill,5,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,0xc7140ddb5ae5dc94d4553fb05d4600816f33ff024844cebe8326f4c41c4a1a47,Yes,BUY,2026-04-16 00:38:16+00:00,0x70f75228c3bd4c3019cbfdade117c968ef57431bd50766855ae7c8665744c64f,0.682,NaN,...,True,0.683000,0.769058,NaN,14.641288,14.641288,14.641288,c,0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,1.660570
3,fill,6,0xff052a89d46a289a32283dfa7815ba23da49fad7,0x0b405dfa1d9ea2a8f0e97b1052ef2b0000f67bddff0242ff4ec406c6e3df5cfa,Yes,BUY,2026-04-16 01:36:34+00:00,0x19e0a85d77e95da819774e6914165231e8c7b45f6f51300f8223ec3aab05928a,0.004,NaN,...,False,0.004471,-0.062661,NaN,14.000000,14.000000,14.000000,0,0xff052a89d46a289a32283dfa7815ba23da49fad7,0.062599
4,fill,8,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,No,BUY,2026-04-16 01:52:00+00:00,0x55e617b3156c9a4f09baeab2d6c7e7b7725ec31acc6bb0418a05f930f0954072,0.850,NaN,...,True,0.850000,0.524657,NaN,3.517646,3.517646,3.517646,6,0x22dbd51e1a4e10fff072fa24300238c64033190f,2.989999


In [205]:
fills_with_wallet[fills_with_wallet['condition_id'] == '0xfdd729d85e44a153ae8d53e709ced74f249815e93570b98c6037ac479ec257b8'].groupby('wallet')['wallet'].count()

Series([], Name: wallet, dtype: int64)

In [206]:
fills_with_wallet[fills_with_wallet['dt'] >= '2026-05-20'].groupby("condition_id")['condition_id'].count().sort_values(ascending=False).head(20)

condition_id
0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423    303
0x6a8cfe84d17693425f27831db5949d7511f3393d4624b182ac6956164cd32b10    244
0x421bc1929df1429cf2cb94f80c1ce6a3ed0d1f0b7a2749b9890075f94eb549e9    219
0xa48ee32a0cbe5bc1a48844bd14e1691701d3bf8f45f4dd8cb8d1d304561393b6    153
0xd8664606498aba2a8ccab56656860cb3f02d74de80e974ab87a9f940e65d4d5e    121
0x3094a2b925483a06aa72945a1472e311e5eb6be75284f61e0c008e279508ddf6    102
0x366f89649caea042c96ee741b185461ec7faa408a2664ec44469a0061924b537     99
0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9f6dc8f59dea51df6c734     95
0xc88e812beb27442554b9231273f6cdda7c96022afc1a5b2a841f03c17dbceaaf     88
0x16608c4fdd7cb41292f6d42c1c02b43ac22593f1c198a104847a3479848d7abe     86
0xf526b2fbff94ae996818e76c8b54cc99ffbfa0a1a9972a48253ada65e32786f7     83
0x9a87a0011494682e4d17a9073059cbbd32ebea4bbc7284219afb5e9a6b4ca8f6     76
0xab1209b73e1ce9ebb4905dad781496385b5102c048889502b174ed733de1506a     71
0xfedc055699e34eac5f8cd6a

In [207]:
from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & (mdf['winner_token_id'].notna())
    ].copy().reset_index(drop=True)


In [208]:
mdf.head()

,condition_id,end_date_iso,question,tags,primary_tag,winner_token_id,outcome
0,0x055176c9ebd8775c281ad540d5c16b3323e316507aef2d45c84f061cbc6bbcdc,2022-08-21T00:00:00Z,"MLB: Who will win Boston Red Sox v. Baltimore Orioles, scheduled for August 21, 7:10 PM ET?",[All],All,9612890763764062692282935414227141810568206972440321500296202304471805951204,Orioles
1,0x1409177eae7f1b0406bc0eea9d2715c9f76d88ec7765aed03cf39d59f36008f7,2023-01-06T00:00:00Z,Will a House Speaker be elected by end of Thursday?,[All],All,56419231299380534710656783098291368308638365867243518354348835265264811381897,No
2,0xe7d21325e5cfccbdaddf6ab6ef9bf477cf3aa6615dd36c8c408da99a1dd83237,2023-01-14T00:00:00Z,Will Volodymyr Zelenskyy be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,5775128592397269721112080310923554806977502333079078864830955469822139503940,No
3,0xdb4b110117e21b9f4505b969dac4790b883315c5782d437eb27e256deaa88d7a,2023-01-14T00:00:00Z,Will Elon Musk be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,111925628487742883146996804476220726156559363451635893056065831198346085566170,No
4,0xd6165c1a30269d0799b27ee6ea90c048dfd6b27276cbd2aa7e18fe1c4562faa8,2023-01-14T00:00:00Z,Will Benjamin Netanyahu be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,115040502429502340199300991115762950152511906206476760034830683863926486402417,No


## Browser plots: PnL and positions

Interactive Plotly charts for cumulative PnL and signed positions. Market hovers include `end_date_iso` from `mdf`.


In [209]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "browser"

TOP_WALLETS_PNL = 10
TOP_MARKETS_PNL = 10
TOP_WALLETS_POSITION = 10
TOP_MARKETS_POSITION = 10

def _short_wallet(wallet: str) -> str:
    wallet = str(wallet)
    return f"{wallet[:6]}...{wallet[-4:]}" if len(wallet) > 14 else wallet

def _short_text(text: str, width: int = 72) -> str:
    if pd.isna(text):
        return "Unknown market"
    text = " ".join(str(text).split())
    return text if len(text) <= width else text[: width - 3] + "..."

def _build_total_ts(df: pd.DataFrame, value_col: str, net_col: str, cum_col: str) -> pd.DataFrame:
    total = (
        df.sort_values("dt")
        .groupby("dt", as_index=False)[value_col]
        .sum()
        .rename(columns={value_col: net_col})
    )
    total[cum_col] = total[net_col].cumsum()
    return total

def _build_position_snapshots(df: pd.DataFrame) -> pd.DataFrame:
    series_cols = [
        "wallet",
        "wallet_label",
        "condition_id",
        "market_label",
        "end_date_iso",
        "market_close_dt",
        "outcome",
    ]

    active = df[
        df["market_close_dt"].isna() | (df["dt"] < df["market_close_dt"])
    ].copy()

    snapshots = (
        active.sort_values(series_cols + ["dt", "order_id"])
        .groupby(series_cols + ["dt"], as_index=False, dropna=False)
        .agg(
            token_position=("filled_token_by_wallet_position", "last"),
            exec_price=("exec_price", "last"),
        )
    )
    snapshots["usdc_position"] = snapshots["token_position"] * snapshots["exec_price"]

    if snapshots.empty:
        snapshots["token_delta"] = pd.Series(dtype=float)
        snapshots["usdc_delta"] = pd.Series(dtype=float)
        return snapshots

    reset = (
        snapshots.groupby(series_cols, as_index=False, dropna=False)
        .agg(
            token_position=("token_position", "last"),
            usdc_position=("usdc_position", "last"),
        )
    )
    reset = reset[
        reset["market_close_dt"].notna()
        & ((reset["token_position"].abs() > 1e-12) | (reset["usdc_position"].abs() > 1e-12))
    ].copy()
    reset["dt"] = reset["market_close_dt"]
    reset["token_position"] = 0.0
    reset["usdc_position"] = 0.0

    combined = pd.concat(
        [
            snapshots[series_cols + ["dt", "token_position", "usdc_position"]],
            reset[series_cols + ["dt", "token_position", "usdc_position"]],
        ],
        ignore_index=True,
    )
    combined = (
        combined.sort_values(series_cols + ["dt"])
        .drop_duplicates(subset=series_cols + ["dt"], keep="last")
        .reset_index(drop=True)
    )
    combined["token_delta"] = combined.groupby(series_cols, dropna=False)["token_position"].diff()
    combined["usdc_delta"] = combined.groupby(series_cols, dropna=False)["usdc_position"].diff()
    combined["token_delta"] = combined["token_delta"].fillna(combined["token_position"])
    combined["usdc_delta"] = combined["usdc_delta"].fillna(combined["usdc_position"])
    return combined

market_meta = (
    mdf[["condition_id", "question", "end_date_iso", "primary_tag"]]
    .drop_duplicates(subset=["condition_id"])
    .copy()
)
market_meta["end_date"] = pd.to_datetime(market_meta["end_date_iso"], utc=True, errors="coerce")
market_meta["market_close_dt"] = market_meta["end_date"].dt.floor("D") + pd.Timedelta(days=1)

plot_fills = fills.copy()
plot_fills["dt"] = pd.to_datetime(plot_fills["dt"], utc=True)
plot_fills = plot_fills.merge(market_meta, on="condition_id", how="left")
plot_fills["wallet_label"] = plot_fills["wallet"].map(_short_wallet)
plot_fills["market_label"] = plot_fills["question"].map(_short_text)
plot_fills["market_label"] = np.where(
    plot_fills["market_label"].eq("Unknown market"),
    plot_fills["condition_id"].str[:12],
    plot_fills["market_label"],
)

pnl_total_ts = _build_total_ts(plot_fills, "fill_pnl_usdc", "net_pnl_usdc", "cum_pnl_usdc")

wallet_pnl_totals = (
    plot_fills.groupby(["wallet", "wallet_label"], as_index=False, dropna=False)
    .agg(total_pnl_usdc=("fill_pnl_usdc", "sum"))
)
top_wallets_pnl = wallet_pnl_totals.reindex(
    wallet_pnl_totals["total_pnl_usdc"].abs().sort_values(ascending=False).index
).head(TOP_WALLETS_PNL).copy()
wallet_pnl_ts = (
    plot_fills[plot_fills["wallet"].isin(top_wallets_pnl["wallet"])]
    .groupby(["wallet", "wallet_label", "dt"], as_index=False, dropna=False)
    .agg(net_pnl_usdc=("fill_pnl_usdc", "sum"))
    .sort_values(["wallet", "dt"])
)

plot_fills["fill_pnl_usdc"] = pd.to_numeric(plot_fills["fill_pnl_usdc"], errors="coerce").fillna(0.0)
wallet_pnl_ts["cum_pnl_usdc"] = plot_fills["fill_pnl_usdc"]

market_pnl_totals = (
    plot_fills.groupby(["condition_id", "market_label", "end_date_iso"], as_index=False, dropna=False)
    .agg(total_pnl_usdc=("fill_pnl_usdc", "sum"))
)
top_markets_pnl = market_pnl_totals.reindex(
    market_pnl_totals["total_pnl_usdc"].abs().sort_values(ascending=False).index
).head(TOP_MARKETS_PNL).copy()
market_pnl_ts = (
    plot_fills[plot_fills["condition_id"].isin(top_markets_pnl["condition_id"])]
    .groupby(["condition_id", "market_label", "end_date_iso", "dt"], as_index=False, dropna=False)
    .agg(net_pnl_usdc=("fill_pnl_usdc", "sum"))
    .sort_values(["condition_id", "dt"])
)
market_pnl_ts["cum_pnl_usdc"] = market_pnl_ts.groupby("condition_id", dropna=False)["net_pnl_usdc"].cumsum()

granular_position_ts = _build_position_snapshots(plot_fills)

wallet_position_all = (
    granular_position_ts.groupby(["wallet", "wallet_label", "dt"], as_index=False, dropna=False)
    .agg(
        net_token_position=("token_delta", "sum"),
        net_usdc_position=("usdc_delta", "sum"),
    )
    .sort_values(["wallet", "dt"])
)
wallet_position_all["cum_token_position"] = wallet_position_all.groupby("wallet", dropna=False)["net_token_position"].cumsum()
wallet_position_all["cum_usdc_position"] = wallet_position_all.groupby("wallet", dropna=False)["net_usdc_position"].cumsum()
wallet_position_rank = (
    wallet_position_all.groupby(["wallet", "wallet_label"], as_index=False, dropna=False)
    .agg(max_abs_usdc_position=("cum_usdc_position", lambda s: s.abs().max()))
)
top_wallets_position = wallet_position_rank.sort_values("max_abs_usdc_position", ascending=False).head(TOP_WALLETS_POSITION).copy()
wallet_position_ts = wallet_position_all[wallet_position_all["wallet"].isin(top_wallets_position["wallet"])].copy()

market_position_all = (
    granular_position_ts.groupby(["condition_id", "market_label", "end_date_iso", "dt"], as_index=False, dropna=False)
    .agg(
        net_token_position=("token_delta", "sum"),
        net_usdc_position=("usdc_delta", "sum"),
    )
    .sort_values(["condition_id", "dt"])
)
market_position_all["cum_token_position"] = market_position_all.groupby("condition_id", dropna=False)["net_token_position"].cumsum()
market_position_all["cum_usdc_position"] = market_position_all.groupby("condition_id", dropna=False)["net_usdc_position"].cumsum()
market_position_rank = (
    market_position_all.groupby(["condition_id", "market_label", "end_date_iso"], as_index=False, dropna=False)
    .agg(max_abs_usdc_position=("cum_usdc_position", lambda s: s.abs().max()))
)
top_markets_position = market_position_rank.sort_values("max_abs_usdc_position", ascending=False).head(TOP_MARKETS_POSITION).copy()
market_position_ts = market_position_all[market_position_all["condition_id"].isin(top_markets_position["condition_id"])].copy()

token_total_ts = _build_total_ts(granular_position_ts, "token_delta", "net_token_position", "cum_token_position")
usdc_total_ts = _build_total_ts(granular_position_ts, "usdc_delta", "net_usdc_position", "cum_usdc_position")

print(
    f"Prepared plot datasets: total pnl={len(pnl_total_ts)}, "
    f"wallet pnl series={wallet_pnl_ts['wallet'].nunique()}, "
    f"market pnl series={market_pnl_ts['condition_id'].nunique()}, "
    f"wallet position series={wallet_position_ts['wallet'].nunique()}, "
    f"market position series={market_position_ts['condition_id'].nunique()}"
)


Prepared plot datasets: total pnl=9879, wallet pnl series=10, market pnl series=10, wallet position series=10, market position series=10


In [210]:
fig_pnl = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total cumulative PnL",
        f"Per-wallet cumulative PnL (top {TOP_WALLETS_PNL} by abs total PnL)",
        f"Per-market cumulative PnL (top {TOP_MARKETS_PNL} by abs total PnL)",
    ),
)

fig_pnl.add_trace(
    go.Scatter(
        x=pnl_total_ts["dt"],
        y=pnl_total_ts["cum_pnl_usdc"],
        mode="lines",
        name="Total",
        line=dict(width=3, color="#1f77b4"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Total cum PnL: %{y:,.2f} USDC<extra></extra>",
    ),
    row=1,
    col=1,
)

wallet_totals_map = top_wallets_pnl.set_index("wallet")["total_pnl_usdc"].to_dict()
for wallet in top_wallets_pnl["wallet"]:
    sub = wallet_pnl_ts[wallet_pnl_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    total = wallet_totals_map[wallet]
    fig_pnl.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_pnl_usdc"],
            mode="lines",
            name=f"{label} ({total:,.1f})",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Cum PnL: %{y:,.2f} USDC<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

market_totals_map = top_markets_pnl.set_index("condition_id")["total_pnl_usdc"].to_dict()
for condition_id in top_markets_pnl["condition_id"]:
    sub = market_pnl_ts[market_pnl_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    total = market_totals_map[condition_id]
    fig_pnl.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_pnl_usdc"],
            mode="lines",
            name=f"{label} ({total:,.1f})",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Cum PnL: %{y:,.2f} USDC<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_pnl.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_pnl.update_yaxes(title_text="USDC", row=1, col=1)
fig_pnl.update_yaxes(title_text="USDC", row=2, col=1)
fig_pnl.update_yaxes(title_text="USDC", row=3, col=1)
fig_pnl.update_xaxes(title_text="Time", row=3, col=1)
fig_pnl.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - cumulative PnL",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_pnl.show(renderer="browser")


In [211]:
fig_token_pos = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total signed token position",
        f"Per-wallet signed token position (top {TOP_WALLETS_POSITION} by max abs USDC position)",
        f"Per-market signed token position (top {TOP_MARKETS_POSITION} by max abs USDC position)",
    ),
)

fig_token_pos.add_trace(
    go.Scatter(
        x=token_total_ts["dt"],
        y=token_total_ts["cum_token_position"],
        mode="lines",
        line_shape="hv",
        name="Total token position",
        line=dict(width=3, color="#2ca02c"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Signed token position: %{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

for wallet in top_wallets_position["wallet"]:
    sub = wallet_position_ts[wallet_position_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_token_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_token_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f} USDC)",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed token position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

for condition_id in top_markets_position["condition_id"]:
    sub = market_position_ts[market_position_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_token_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_token_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f} USDC)",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed token position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_token_pos.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_token_pos.update_yaxes(title_text="Tokens", row=1, col=1)
fig_token_pos.update_yaxes(title_text="Tokens", row=2, col=1)
fig_token_pos.update_yaxes(title_text="Tokens", row=3, col=1)
fig_token_pos.update_xaxes(title_text="Time", row=3, col=1)
fig_token_pos.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - signed token positions",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_token_pos.show(renderer="browser")


In [212]:
fig_usdc_pos = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total signed USDC position (using traded price)",
        f"Per-wallet signed USDC position (top {TOP_WALLETS_POSITION} by max abs USDC position)",
        f"Per-market signed USDC position (top {TOP_MARKETS_POSITION} by max abs USDC position)",
    ),
)

fig_usdc_pos.add_trace(
    go.Scatter(
        x=usdc_total_ts["dt"],
        y=usdc_total_ts["cum_usdc_position"],
        mode="lines",
        line_shape="hv",
        name="Total USDC position",
        line=dict(width=3, color="#d62728"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Signed USDC position: %{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

for wallet in top_wallets_position["wallet"]:
    sub = wallet_position_ts[wallet_position_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_usdc_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_usdc_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f})",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed USDC position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

for condition_id in top_markets_position["condition_id"]:
    sub = market_position_ts[market_position_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_usdc_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_usdc_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f})",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed USDC position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_usdc_pos.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_usdc_pos.update_yaxes(title_text="USDC", row=1, col=1)
fig_usdc_pos.update_yaxes(title_text="USDC", row=2, col=1)
fig_usdc_pos.update_yaxes(title_text="USDC", row=3, col=1)
fig_usdc_pos.update_xaxes(title_text="Time", row=3, col=1)
fig_usdc_pos.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - signed USDC positions",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_usdc_pos.show(renderer="browser")
